#LoRA pathway
Text data -> datasets -> tokenizer -> Transformers Model -> Peft (LoRA) -> Trainer

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig

Step 1: Load base model and tokenizer

In [6]:
model_name='gpt2'
model=AutoModelForCausalLM.from_pretrained(model_name)
tokenizer=AutoTokenizer.from_pretrained(model_name)


if tokenizer.pad_token is None:
  tokenizer.pad_token=tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Step 2: define LoRA configuration

In [10]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['c_attn']
)

#wrap model with Lora Adapter

model=get_peft_model(model,lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(


Step 3: Create a tiny synthetic dataset

In [13]:
train_texts=['Hello world','Lora fine tunning is fun','Transfer Learning']
eval_texts=['Testing evaluation','Another sample']

train_dataset=Dataset.from_dict({'text': train_texts})
eval_dataset=Dataset.from_dict({'text': eval_texts})


def tokenize_func(examples):
  tokens=tokenizer(
      examples['text'],truncation=True, padding='max_length',max_length=64
  )
  tokens['labels']=tokens['input_ids'].copy()
  return tokens


# Apply tokenization
train_dataset = train_dataset.map(tokenize_func, batched=True)
eval_dataset = eval_dataset.map(tokenize_func, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Step 4. Training configuration

In [14]:
training_args = TrainingArguments(
    output_dir="./lora-finetuned-model",   # save directory
    per_device_train_batch_size=2,         # small batch size
    gradient_accumulation_steps=2,         # accumulate gradients
    learning_rate=2e-4,                    # tuned for LoRA
    num_train_epochs=1,                    # demo run
    logging_steps=1,                       # log every step
    save_strategy="epoch",                 # save at end of epoch
    fp16=True                              # mixed precision training
)


Step 5: initialize Trainer

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

Step 6: Train

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,7.562579


TrainOutput(global_step=1, training_loss=7.5625786781311035, metrics={'train_runtime': 8.8211, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.113, 'total_flos': 98324250624.0, 'train_loss': 7.5625786781311035, 'epoch': 1.0})

Step 7. Save only LoRA adapter weights (few MB instead of GBs!)

In [17]:
model.save_pretrained("./lora-weights")

print("Training complete. LoRA weights saved in ./lora-weights")

Training complete. LoRA weights saved in ./lora-weights
